In [0]:
# Change the course name as per your course name
COURSE_NAME = "AI BI for Data Analysts"
RUN_QA_CHECKER = True
QA_TASK_RELATIVE_PATH = "./qa_content_checker"
QA_TASK_NAME = "QA Content Checker — Grammar, Deprecated, UI Steps"

# Tester emails — notified on job failure and success. Add as many as you like with comma seperated emails
TESTER_EMAILS = [
    "gourav.prajapati@databricks.com"
]

# Course folder — this runner lives two levels below the course folder
# (.../AI BI for Data Analysts/Includes/automation/), so we need "../../"
# to reach the course root from the automation directory.
_PKG = "../.."

# No lab notebooks need auto-filling: this course has NO <FILL_IN> placeholders anywhere
# (verified across all 15 notebooks). The two "Lab" notebooks (05, 11) are dashboard-build
# UI exercises whose only executable cells are a %run setup + a preview query, so there is
# nothing to fill. Leave empty.
LAB_NOTEBOOKS = []

# ── Compute Type Registry ─────────────────────────────────────────────────────
# Define all compute configurations available to tasks in this course.
# Each key becomes the compute_type value you assign to a task in COURSE_TASKS.
# Cell 14 reads these entries to create the correct compute resource per task.
#
# Supported "type" values:
#   "serverless"      – Databricks Serverless. Requires: environment_version (str)
#   "classic_cluster" – Existing all-purpose cluster. Requires: cluster_id (str)
#   "new_cluster"     – New cluster per run. Requires: cluster_config (dict)
#                       Keys map directly to ClusterSpec fields:
#                       spark_version, node_type_id, num_workers, etc.
#
# Tasks that omit compute_type fall back to DEFAULT_COMPUTE_TYPE.
# ─────────────────────────────────────────────────────────────────────────────

DEFAULT_COMPUTE_TYPE = "serverless_v5"

COMPUTE_TYPES = {
    "serverless_v5": {
        "type": "serverless",
        "environment_version": "5",
    },
    # ── SQL Warehouse ─────────────────────────────────────────────────────────
    # Use for notebooks that require a SQL warehouse (e.g. SQL Scripting,
    # Stored Procedures). The job will find an existing warehouse by name or
    # create one automatically using the config below.
    # Only PRO / Serverless warehouses are supported — not Classic.
    "shared_warehouse": {
        "type": "warehouse",
        "warehouse_name": "shared_warehouse",
        "cluster_size": "2X-Small",
        "warehouse_type": "PRO",
        "auto_stop_mins": 30,
    },
    # "serverless_v4": {
    #     "type": "serverless",
    #     "environment_version": "4",
    # },
    # "my_classic_cluster": {
    #     "type": "classic_cluster",
    #     "cluster_id": "<your-cluster-id>",
    # },
    # "new_standard_cluster": {
    #     "type": "new_cluster",
    #     "cluster_config": {
    #         "spark_version": "15.4.x-scala2.12",
    #         "node_type_id":  "i3.xlarge",
    #         "num_workers":   2,
    #     },
    # },
}

# ── Task Retry Policy ─────────────────────────────────────────────────────────
# Controls automatic retries for failed tasks. Helps handle transient issues
# like Unity Catalog metadata propagation delays after the setup task creates
# schemas/tables that aren't immediately visible to downstream tasks.
#
# Set TASK_MAX_RETRIES = 0 to disable retries entirely.
# Per-task override: add "max_retries" and/or "retry_interval_seconds" keys
# to any entry in COURSE_TASKS to override the global defaults for that task.
# ─────────────────────────────────────────────────────────────────────────────
TASK_MAX_RETRIES = 2                    # Number of retries on failure (0 = no retries)
TASK_RETRY_INTERVAL_SECONDS = 30        # Seconds to wait between retry attempts (global default)
WAREHOUSE_RETRY_INTERVAL_SECONDS = 45   # Seconds for warehouse tasks (cross-compute metadata sync is slower)

# ── Course notebook DAG ───────────────────────────────────────────────────────
# Classification (by executable-cell content + language). This is a dashboard-focused
# course — most notebooks are pure UI walk-throughs (0 executable cells). Only the ones
# with real runnable code become tasks.
#
#   RUN AS NOTEBOOK TASKS:
#     required_setup  01 Demo - REQUIRED - Data Setup   serverless        (%run Classroom-Setup-Required-Data (Python): CREATE VOLUMEs, copy CSVs, CREATE TABLE for all ca_* + zip_codes + nyctaxi_gold; then 11 SQL preview/aggregation queries)
#     demo_03         03 Demo - Designing Datasets       shared_warehouse  (only exec cell is %run Classroom-Setup-ca_setup, which verifies ca_* tables + sets catalog/schema)
#     demo_04b        04B Demo - Province Breakdowns     shared_warehouse  (%run ca_setup + a DECLARE/SET/SELECT image-URL cell; runs clean, just emits a placeholder string)
#     lab_05          05 Lab - Build a Dashboard (Taxi)  shared_warehouse  (%run Classroom-Setup-lab-setup verifies nyctaxi_gold + a SELECT preview)
#     demo_09         09 Demo - Running in Production     serverless        (%run ca_setup + a Python cell + INSERT rows into ca_orders to show refresh; .ipynb w/ Python → serverless)
#     lab_11          11 Lab - Extend & Share Taxi Dash  serverless        (%run Lab-Starter-Dashboard-Creation → uses the SDK to import a .lvdash.json dashboard; Python → serverless)
#
#   COMPUTE RULE ([[feedback_compute_type_nonsql]]): any notebook whose default language
#   is Python, or that contains %python / bare .ipynb Python code cells / dbutils / the
#   SDK, MUST run on serverless, NOT a SQL warehouse. required_setup (Python setup),
#   09 (has a Python cell) and 11 (SDK dashboard import) → serverless. 03/04B/05 are pure
#   SQL (their setups %run the SQL-language Common-SQL-WAREHOUSE) → shared_warehouse.
#
#   EXCLUDED — UI-driven / lecture / not headless-runnable (all have 0 executable cells
#   unless noted; verified none create data other notebooks depend on):
#     00 - Course Overview           lecture/overview, 0 code cells.
#     02 Lecture - Intro to AIBI      lecture, 0 code cells.
#     04A Demo - KPIs & Yearly Trends UI: build KPI/trend viz in the dashboard, 0 code cells.
#     06 Demo - Genie Code (Page 2)   UI: Genie-drafted viz in the dashboard, 0 code cells.
#     07 Demo - Adding Filters        UI: add filters in the dashboard, 0 code cells.
#     08 Demo - Publishing & Sharing  UI: publish/share the dashboard, 0 code cells.
#     10 Lecture - Summary            lecture, 0 code cells.
#     Appendix A - Dashboard Caching  UI/conceptual, 0 code cells.
#     WIP - PuPr - Appendix B         WIP local metric views, 0 code cells.
#     Final Demonstration Dashboard/  instructor asset (builds the solution dashboard), not a lesson.
#
# DATA PREREQUISITE: `01 Demo - REQUIRED - Data Setup` %runs Classroom-Setup-Required-Data
# (Python) → Classroom-Setup-Common-Python, which provisions a per-user UC catalog, the
# `aibi_dashboards` + taxi schemas, `source_files`/`business_icon` volumes, copies the CSVs,
# and CREATEs every table (ca_orders/ca_customers/ca_products/ca_opportunities/
# ca_sales_targets, zip_codes, nyctaxi_gold). It MUST run first. The demo/lab setups
# (Classroom-Setup-ca_setup / -lab-setup, both SQL) do NOT create tables — they CALL a
# check_required_tables procedure that RAISES if a table is missing, then set the catalog/
# schema. So every downstream task hard-depends on required_setup.
#
# DEPENDENCIES: all tasks depend only on required_setup and run in parallel afterward. They
# read the ca_*/taxi tables read-only, except 09 which INSERTs a handful of rows into
# ca_orders (demonstrating a refresh) — no other task mutates ca_orders, so no conflict.
# Cross-compute metadata propagation (serverless setup → warehouse tasks) is covered by the
# task retry policy below.
COURSE_TASKS = [
    {
        "task_key": "required_setup",
        "name": "01 Demo - REQUIRED - Data Setup and Exploration (creates all ca_*/taxi tables)",
        "relative_path": f"{_PKG}/01 Demo - REQUIRED - Data Setup and Exploration",
        "depends_on": [],
        "compute_type": "serverless_v5",   # Python setup: CREATE VOLUME, copy CSVs, CREATE TABLE
    },
    {
        "task_key": "demo_03",
        "name": "03 Demo - Designing Datasets for Dashboards",
        "relative_path": f"{_PKG}/03 Demo - Designing Datasets for Dashboards",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # only exec cell is %run ca_setup (SQL)
    },
    {
        "task_key": "demo_04b",
        "name": "04B Demo - Page 1 Visualizations - Province Breakdowns",
        "relative_path": f"{_PKG}/04B Demo - Page 1 Visualizations - Province Breakdowns",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # %run ca_setup + DECLARE/SET/SELECT image-URL (SQL)
    },
    {
        "task_key": "lab_05",
        "name": "05 Lab - Build a Dashboard - NYC Taxi Data",
        "relative_path": f"{_PKG}/05 Lab - Build a Dashboard - NYC Taxi Data",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # %run lab-setup + SELECT preview (SQL)
    },
    {
        "task_key": "demo_09",
        "name": "09 Demo - Running Your Dashboard in Production",
        "relative_path": f"{_PKG}/09 Demo - Running Your Dashboard in Production",
        "depends_on": ["required_setup"],
        "compute_type": "serverless_v5",   # .ipynb has a Python cell + INSERT INTO ca_orders
    },
    {
        "task_key": "lab_11",
        "name": "11 Lab - Extend and Share Your Taxi Dashboard",
        "relative_path": f"{_PKG}/11 Lab - Extend and Share Your Taxi Dashboard",
        "depends_on": ["required_setup"],
        "compute_type": "serverless_v5",   # %run Lab-Starter → SDK dashboard import (Python)
    },
]

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: wrap_failing_sql_cell / restore_sql_cell
#
# Purpose
# -------
# Some demo notebooks contain SQL cells that are *intentionally designed to fail*
# (e.g. to demonstrate BEGIN...END halting behaviour). When those notebooks run
# as job tasks the cell failure stops the entire notebook, skipping all
# subsequent cells (cleanup, further demos, etc.).
#
# These helpers let the course tester temporarily wrap any such cell with an
# outer BEGIN...DECLARE EXIT HANDLER...END block — pure SQL, no language change
# — so the notebook continues past the intentional failure. After the job
# completes the original source is restored so students always see the clean
# demo notebook.
#
# Identifying the target cell
# ----------------------------
# Cells are identified by cell_index: the 1-based position of the cell within
# the notebook (counting ALL cells, markdown included). Unlike cell_nuid —
# which regenerates every time a lab notebook is re-provisioned/reset — a
# cell's position in an unchanged template stays the same across resets, so
# cell_index remains valid as long as no cells are added/removed above the
# target cell.
#
# How to use
# ----------
# 1. Before job creation / run:
#       restores = patch_notebooks_for_job(CELLS_TO_PATCH)
#
# 2. After job completes (pass/fail):
#       restore_notebooks_after_job(restores)
#
# CELLS_TO_PATCH is a list of dicts defined in the cell that follows this one.
# Each entry:
#   {
#       "notebook_path"  : absolute workspace path to the notebook,
#       "cell_index"     : 1-based position of the SQL cell to wrap,
#       "handler_message": (optional) text returned by the handler SELECT,
#   }
# ─────────────────────────────────────────────────────────────────────────────

import base64, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

_sdk_client = None

def _get_client(w=None):
    """Returns a WorkspaceClient, reusing a module-level instance if available."""
    if w:
        return w
    global _sdk_client
    if _sdk_client is None:
        _sdk_client = WorkspaceClient()
    return _sdk_client


def _get_cell_source(cell):
    """Returns a cell's source as a single string, regardless of list/str form."""
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source


def wrap_failing_sql_cell(
    notebook_path: str,
    cell_index: int,
    handler_message: str = None,
    w=None,
) -> str:
    """
    Wraps a SQL cell's BEGIN...END block with an outer DECLARE EXIT HANDLER
    so the notebook continues executing even if the inner block fails.
    The cell stays SQL — no language change.

    Args:
        notebook_path   : Absolute workspace path to the target notebook.
        cell_index      : 1-based position of the cell to wrap (counting ALL
                          cells, markdown included).
        handler_message : Optional message shown as a result row when the
                          exception is caught.
        w               : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — pass this to restore_sql_cell() to undo.
    """
    _w  = _get_client(w)
    msg = handler_message or (
        "[Expected failure] The inner BEGIN...END block failed as designed. "
        "Notebook continues."
    )

    # --- export
    export  = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb      = json.loads(base64.b64decode(export.content))
    cells   = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cell = cells[cell_index - 1]

    original_source = _get_cell_source(cell)
    indented = "\n".join("  " + line for line in original_source.splitlines())
    cell["source"] = (
        "-- Outer block catches the expected error; inner block is unchanged.\n"
        "BEGIN\n"
        "  DECLARE EXIT HANDLER FOR SQLEXCEPTION\n"
        f"    SELECT '{msg}' AS message;\n\n"
        "  BEGIN\n"
        f"{indented}\n"
        "  END;\n"
        "END;"
    )

    # --- re-import
    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Wrapped  : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_sql_cell(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses wrap_failing_sql_cell().
    Use the SAME cell_index that was passed to wrap_failing_sql_cell().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by wrap_failing_sql_cell().
        w              : WorkspaceClient to reuse (optional).
    """
    _w     = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb     = json.loads(base64.b64decode(export.content))
    cells  = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_notebooks_for_job(patches: list, w=None) -> list:
    """
    Wraps all cells listed in `patches` before a job run.
    Returns a restore-list to pass to restore_notebooks_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path   (str) — absolute workspace path
                    cell_index      (int) — 1-based position of the failing cell
                    handler_message (str, opt) — custom handler message
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    print("\n═" * 36 + "\nPATCHING NOTEBOOKS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = wrap_failing_sql_cell(
            notebook_path   = p["notebook_path"],
            cell_index      = p["cell_index"],
            handler_message = p.get("handler_message"),
            w               = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_notebooks_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_notebooks_for_job().

    Args:
        restores : the list returned by patch_notebooks_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    print("\n═" * 36 + "\nRESTORING NOTEBOOKS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_sql_cell(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")



# ─────────────────────────────────────────────────────────────────────────────
# replace_cell_text / restore_cell_text
#
# Purpose
# -------
# Generic text-replacement helpers: swap a specific substring in a notebook
# cell's source before a job run, then put it back afterward.
#
# Useful when a single line in an already-filled notebook needs a small tweak
# for automated testing (e.g. converting an intentional single-statement
# failure into a no-op) without wrapping the whole cell in a
# BEGIN...DECLARE EXIT HANDLER block.
#
# Four parameters
# ---------------
#   notebook_path    — absolute workspace path to the notebook
#   cell_index       — 1-based position of the cell (counting ALL cells)
#   original_text    — the exact text to find (can span multiple lines)
#   replacement_text — the text to substitute in
# ─────────────────────────────────────────────────────────────────────────────

def replace_cell_text(
    notebook_path: str,
    cell_index: int,
    original_text: str,
    replacement_text: str,
    w=None,
) -> str:
    """
    Replaces a specific text substring in a notebook cell's source.

    Args:
        notebook_path    : Absolute workspace path to the target notebook.
        cell_index       : 1-based position of the cell to modify (counting ALL cells).
        original_text    : The exact text to find and replace.
        replacement_text : The text to substitute in.
        w                : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — the full original cell source before replacement.
                                Pass to restore_cell_text() to undo.
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )

    cell = cells[cell_index - 1]
    original_source = _get_cell_source(cell)

    if original_text not in original_source:
        raise ValueError(
            f"original_text not found in cell {cell_index} of "
            f"{notebook_path.split('/')[-1]!r}.\n"
            f"original_text: {original_text!r}"
        )

    cell["source"] = original_source.replace(original_text, replacement_text, 1)

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Replaced : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_cell_text(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses replace_cell_text().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by replace_cell_text().
        w              : WorkspaceClient to reuse (optional).
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_text_replacements_for_job(patches: list, w=None) -> list:
    """
    Applies targeted text replacements to notebook cells before a job run.
    Returns a restore-list to pass to restore_text_replacements_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path    (str) — absolute workspace path
                    cell_index       (int) — 1-based position of the cell
                    original_text    (str) — exact text to replace
                    replacement_text (str) — text to substitute in
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    if not patches:
        print("  (no text replacements configured)\n")
        return []
    print("\n" + "═" * 36 + "\nAPPLYING TEXT REPLACEMENTS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = replace_cell_text(
            notebook_path    = p["notebook_path"],
            cell_index       = p["cell_index"],
            original_text    = p["original_text"],
            replacement_text = p["replacement_text"],
            w                = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_text_replacements_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_text_replacements_for_job().

    Args:
        restores : the list returned by patch_text_replacements_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    if not restores:
        print("  (nothing to restore)\n")
        return
    print("\n" + "═" * 36 + "\nRESTORING TEXT REPLACEMENTS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_cell_text(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")


print("✅ Utility functions loaded: wrap_failing_sql_cell, restore_sql_cell,"
      " patch_notebooks_for_job, restore_notebooks_after_job,"
      " replace_cell_text, restore_cell_text,"
      " patch_text_replacements_for_job, restore_text_replacements_after_job")